In [ ]:
import requests
import pandas as pd
import io
import json

# ==============================================================================
# CONFIGURATION FOR MULTIPLE DATASETS
# ==============================================================================
TARGET_DATASETS = [
    {
        "file_name": "OECD_RD_GDP_2000_2025.csv",
        "agency_id": "OECD.STI.STP",
        "dataflow_id": "DSD_MSTI@DF_MSTI",
        "filters": {
            "UNIT_MEASURE": "PT_B1GQ",
            "MEASURE": "G" 
        }
    }
]

START_PERIOD = "2000"
END_PERIOD = "2025"
VERSION = "all"
DIMENSIONS = "all"

headers = {
    "Accept": "application/vnd.sdmx.data+csv; charset=utf-8; version=2",
    "User-Agent": "DataExtractionScript/3.0 (Python/Requests)"
}

# ==============================================================================
# BATCH EXTRACTION PROCESS
# ==============================================================================

for dataset in TARGET_DATASETS:
    print(f"Processing: {dataset['file_name']}...")
    
    url = f"https://sdmx.oecd.org/public/rest/data/{dataset['agency_id']},{dataset['dataflow_id']},{VERSION}/{DIMENSIONS}"
    
    params = {
        "startPeriod": START_PERIOD,
        "endPeriod": END_PERIOD,
        "dimensionAtObservation": "AllDimensions",
        "format": "csvfilewithlabels"
    }
    
    response = requests.get(url, params=params, headers=headers)
    
    if response.status_code == 200:
        csv_stream = io.StringIO(response.text)
        df = pd.read_csv(csv_stream, low_memory=False)
        df_clean = df.dropna(subset=['OBS_VALUE']).copy()
        
        # Apply the specific filters requested to isolate a single metric
        for column_name, filter_value in dataset.get('filters', {}).items():
            if column_name in df_clean.columns:
                df_clean = df_clean[df_clean[column_name] == filter_value]
            else:
                print(f"  -> Warning: Column '{column_name}' not found.")
        
        if df_clean.empty:
            print(f"  -> Error: Dataset is empty after applying filters.")
            
            debug_dict = {}
            for col in df.columns:
                if col not in ['OBS_VALUE', 'TIME_PERIOD', 'REF_AREA']:
                    debug_dict[col] = df[col].dropna().unique().tolist()
            
            with open("DEBUG_OECD_CODES.txt", "w") as f:
                f.write(json.dumps(debug_dict, indent=4))
                
            print("  -> Debug file generated: 'DEBUG_OECD_CODES.txt'. Please check the exact codes.")
            continue
        
        # Target columns based on SDMX 2.1 standard
        target_columns = ['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']
        
        if all(col in df_clean.columns for col in target_columns):
            df_tidy = df_clean[target_columns].rename(columns={
                'REF_AREA': 'Country',
                'TIME_PERIOD': 'Year',
                'OBS_VALUE': 'Value'
            })
            
            df_tidy = df_tidy.reset_index(drop=True)
            df_tidy.to_csv(dataset['file_name'], index=False)
            
            print(f"  -> Success! Saved {len(df_tidy)} rows to {dataset['file_name']}.\n")
        else:
            print(f"  -> Structural error: Missing SDMX columns in {dataset['file_name']}.\n")
    else:
        print(f"  -> HTTP Error {response.status_code} for {dataset['file_name']}.\n")

Processing: OECD_RD_GDP_2000_2025.csv...
  -> Error: Dataset is empty after applying filters.
  -> Debug file generated: 'DEBUG_OECD_CODES.txt'. Please check the exact codes.
